# *Physarum polycephalum*

Preprocesamiento del experimento de *Physarum polycephalum* con el dataset de Harrington y Harrington (Zenodo,
DOI [10.5281/zenodo.22868368](https://doi.org/10.5281/zenodo.22868368), CC BY 4.0).

Son 24 cajas de Petri de 90 mm (experimentos 001, 002, 008 y 009, con 6 cajas cada uno) escaneadas desde abajo cada
5 min (001, 002) o 2 min (008, 009) durante 63–141 h: 51 036 imágenes RGB de ~1169×1276 px (0.085 mm/px). Cada caja es
un archivo ZIP sin compresión con un OME-Zarr v3 (`s0` = imágenes (T, 3, H, W) uint8, `acquisition/timestamp_utc_ns`,
`registration/dish_circle_xyr_px`, …). La avena dentro de la caja es el alimento del estado inicial.

1. **Descarga** de los 24 ZIP (133 GB) a `Dataset/Physarum/zarr/`.
2. **Muestreo**: una imagen cada N minutos, recortada a la caja → `Dataset/Physarum/cada<N>min/<caja>.npz`.
3. **Timelapse** de las 24 cajas → `Dataset/Physarum/timelapse_cada<N>min.mp4`.

In [ ]:
import hashlib
import re
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import requests
from PIL import Image

TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "main.py").exists())
DATA = TRAINING / "Dataset" / "Physarum"
ZARR = DATA / "zarr"
RECORD = "https://zenodo.org/api/records/22868368"

## 1. Descarga

Baja los archivos del registro de Zenodo con `jobs` descargas a la vez (~10 MB/s en total, ~3 h 15 min). Un archivo a
medias se reanuda donde se quedó, así que si se corta basta volver a correr la celda. Cada ZIP se verifica contra
`SHA256SUMS` del dataset y, si coincide, se marca con `<archivo>.ok` para no volver a verificarlo.

In [ ]:
def bajar(url, destino, intentos=20):
    """Descarga url a destino, reanudando lo que ya exista."""
    for intento in range(intentos):
        hecho = destino.stat().st_size if destino.exists() else 0
        try:
            with requests.get(url, headers={"Range": f"bytes={hecho}-"} if hecho else {}, stream=True,
                              timeout=120) as r:
                if r.status_code == 416:                               # ya estaba completo
                    return
                if r.status_code == 429:
                    time.sleep(float(r.headers.get("Retry-After", 30)))
                    continue
                r.raise_for_status()
                with open(destino, "ab" if r.status_code == 206 else "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            return
        except requests.RequestException as e:
            print(f"  {destino.name}: {e!r}, reintento", flush=True)
            time.sleep(30)
    raise RuntimeError(f"no se pudo bajar {url}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while bloque := f.read(1 << 24):
            h.update(bloque)
    return h.hexdigest()


def descargar(jobs=3):
    ZARR.mkdir(parents=True, exist_ok=True)
    archivos = [f["key"] for f in requests.get(RECORD, timeout=60).json()["files"]]
    for k in archivos:                                                 # documentos (README, SHA256SUMS, …)
        if not k.endswith(".ome.zarr.zip") and not (DATA / k).exists():
            bajar(f"{RECORD}/files/{k}/content", DATA / k)
    sumas = dict(reversed(l.split()) for l in (DATA / "SHA256SUMS").read_text().splitlines())

    def uno(k):
        ok = ZARR / f"{k}.ok"
        if ok.exists():
            return f"{k}: ya estaba"
        bajar(f"{RECORD}/files/{k}/content", ZARR / k)
        if sha256(ZARR / k) != sumas[k]:
            return f"{k}: SHA256 NO COINCIDE (borrarlo y volver a correr)"
        ok.touch()
        return f"{k}: OK"

    with ThreadPoolExecutor(jobs) as ex:
        for msg in ex.map(uno, sorted(k for k in archivos if k.endswith(".ome.zarr.zip"))):
            print(msg, flush=True)
    print(f"{len(list(ZARR.glob('*.ok')))}/24 archivos verificados")

In [ ]:
descargar()

## 2. Muestreo

Por cada caja:

1. Toma una imagen cada `every_min` minutos (por el timestamp real, no el nominal).
2. Recorta un cuadrado centrado en el círculo de la caja ajustado en esa imagen (`registration/dish_circle_xyr_px`; la
   cámara se mueve un poco entre imágenes), con el radio mediano de la serie × `margin`, y lo voltea en x (el escáner ve
   la caja desde abajo).
3. Reduce a `size` × `size` y guarda `Dataset/Physarum/cada<every_min>min/<caja>.npz` con `frames` (T, size, size, 3)
   uint8, `t_s` (s desde la primera imagen) e `idx` (índice original).

In [ ]:
def crop(frame: np.ndarray, x: float, y: float, r: int, size: int) -> np.ndarray:
    """frame (3, H, W) → cuadrado de lado 2r centrado en (x, y), relleno con 0 fuera, volteado en x."""
    img = frame.transpose(1, 2, 0)
    H, W = img.shape[:2]
    x0, y0 = int(round(x)) - r, int(round(y)) - r
    out = np.zeros((2 * r, 2 * r, 3), np.uint8)
    sx, sy = max(0, -x0), max(0, -y0)
    ex, ey = min(2 * r, W - x0), min(2 * r, H - y0)
    out[sy:ey, sx:ex] = img[y0 + sy:y0 + ey, x0 + sx:x0 + ex]
    return np.asarray(Image.fromarray(out[:, ::-1]).resize((size, size), Image.LANCZOS))


def muestrear_caja(nombre, every_min=30, size=320, margin=1.06, carpeta=None):
    import zarr

    carpeta = Path(carpeta or DATA / f"cada{every_min:g}min")
    path = carpeta / f"{nombre}.npz"
    if path.exists():
        return f"{nombre}: ya estaba"
    t0 = time.time()
    root = zarr.open_group(zarr.storage.ZipStore(ZARR / f"{nombre}.ome.zarr.zip", mode="r"), mode="r", zarr_format=3)
    s0 = root["s0"]
    ts = root["acquisition/timestamp_utc_ns"][:]
    circ = root["registration/dish_circle_xyr_px"][:]
    t_s = (ts - ts[0]) / 1e9
    # índice de la primera imagen en o después de cada múltiplo de every_min
    marcas = np.arange(0, t_s[-1] + 1, every_min * 60)
    idx = np.unique(np.clip(np.searchsorted(t_s, marcas), 0, len(t_s) - 1))
    r = int(round(np.median(circ[:, 2]) * margin))
    frames = np.stack([crop(s0[int(i)], circ[i, 0], circ[i, 1], r, size) for i in idx])
    carpeta.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, frames=frames, t_s=t_s[idx], idx=idx)
    tmp.rename(path)
    return f"{nombre}: {len(idx)} imágenes en {time.time() - t0:.0f} s"


def muestrear(every_min=30, size=320, margin=1.06, cajas=None, workers=4):
    cajas = cajas or sorted(p.name[:-len(".ome.zarr.zip")] for p in ZARR.glob("*.ome.zarr.zip"))
    with ThreadPoolExecutor(workers) as ex:
        for msg in ex.map(lambda n: muestrear_caja(n, every_min, size, margin), cajas):
            print(msg, flush=True)

In [ ]:
muestrear(every_min=30, size=320)

## 3. Timelapse

Una fila por experimento (001, 002, 008, 009) y una columna por caja (dish0–dish5), con un solo reloj en horas desde la
primera imagen de cada experimento: en cada paso, cada caja muestra su última imagen con t ≤ reloj. Las cajas cuyo
experimento ya terminó se quedan en su última imagen, oscurecidas y con "fin". Escribe un MP4 (H.264 con libopenh264).

In [ ]:
HEADER, LABEL, LEFT = 44, 22, 120
FONT_PATHS = ["/usr/share/fonts/liberation-mono-fonts/LiberationMono-Regular.ttf",
              "/usr/share/fonts/dejavu-sans-fonts/DejaVuSans.ttf"]


def font(size):
    from PIL import ImageFont

    for p in FONT_PATHS:
        if Path(p).exists():
            return ImageFont.truetype(p, size)
    return ImageFont.load_default(size)


def timelapse(folder="cada30min", tile=240, fps=10):
    from PIL import ImageDraw

    series = {}
    for f in sorted((DATA / folder).glob("physarum-*.npz")):
        exp, dish = re.match(r"physarum-(\d+)-dish(\d)", f.stem).groups()
        d = np.load(f)
        tiles = np.stack([np.asarray(Image.fromarray(x).resize((tile, tile), Image.LANCZOS)) for x in d["frames"]])
        series[(exp, int(dish))] = (d["t_s"] / 3600, tiles)
    exps = sorted({e for e, _ in series})
    paso = float(np.median(np.diff(next(iter(series.values()))[0])))
    fin = max(t[-1] for t, _ in series.values())
    relojes = np.arange(0, fin + paso / 2, paso)

    W = LEFT + 6 * tile
    H = HEADER + len(exps) * (tile + LABEL)
    W, H = W + W % 2, H + H % 2
    out = DATA / f"timelapse_{folder}.mp4"
    ffmpeg = subprocess.Popen(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
         "-r", str(fps), "-i", "-", "-c:v", "libopenh264", "-b:v", "6M", "-pix_fmt", "yuv420p", str(out)],
        stdin=subprocess.PIPE)
    f_titulo, f_txt = font(24), font(15)
    for reloj in relojes:
        img = Image.new("RGB", (W, H), (20, 20, 20))
        dr = ImageDraw.Draw(img)
        dr.text((12, 8), f"Physarum polycephalum   {reloj:6.1f} h", fill=(235, 235, 235), font=f_titulo)
        for i, e in enumerate(exps):
            y = HEADER + i * (tile + LABEL)
            dr.text((10, y + tile // 2 - 10), f"exp. {e}", fill=(235, 235, 235), font=f_txt)
            for j in range(6):
                if (e, j) not in series:
                    continue
                t, tiles = series[(e, j)]
                k = int(np.searchsorted(t, reloj + 1e-6) - 1)
                termino = reloj > t[-1] + paso / 2
                cuadro = tiles[max(k, 0)]
                if termino:
                    cuadro = (cuadro * 0.45).astype(np.uint8)
                x = LEFT + j * tile
                img.paste(Image.fromarray(cuadro), (x, y))
                etiqueta = f"dish{j}  " + ("fin" if termino else f"{t[max(k, 0)]:.1f} h")
                dr.text((x + 6, y + tile + 2), etiqueta, fill=(200, 200, 200), font=f_txt)
        ffmpeg.stdin.write(img.tobytes())
    ffmpeg.stdin.close()
    ffmpeg.wait()
    print(f"{out}: {len(relojes)} imágenes, {len(relojes) / fps:.1f} s, {len(series)} cajas")

In [ ]:
timelapse("cada30min")